# Does a fine-tune that learns to count a new letter run through the count the model already had?

*E55 Stage A, reuse line, 2026-10-29. Report: `reports/analyses/2026-10-29_analysis_e55-stage-a.md`, which holds every number and the scoring.*

## The answer

The two checks work as a ground truth on this grammar. Swapping the count in moves the answer on every model that uses it and never on the nulls. Fast fine-tunes never ran through the count of a. The slow fine-tunes with 80% and with all spurious rows ran through it after a thousand steps. By the end the first had learned b and left the old count, and the all-spurious ones read neither, so one filed prediction missed. None was reversed. The count of b decodes from the base model above shuffled labels, narrowly.

## What we were trying to test

`GOALS.md` asks, if no simple map works, whether post-training teaches new capabilities or strengthens existing ones. Until now our labels "existing" and "new" came from VPD itself, so a sceptic reads every result as a fact about VPD. This stage builds labels that need no decomposition, on a task where we choose whether the model already has the capability.

**The task.** Jain, Kirk and Lubana's counting grammar, rebuilt. A sequence holds a prefix, 250 tokens of grammar text over the letters a, b and c, and a query token `=`. The prefix `C a 40` asks how many a's are in the last 40 text tokens; `C b 40` asks the same for b. Every read and every swap happens at `=`.

**The models.** A minGPT of 6 blocks and about 3M parameters, pretrained for 50K steps. Three pretraining mixes fix what each model can do:

| model | asked to count during pretraining | what it can do |
|---|---|---|
| LOW | a almost always, b one time in a thousand | counts a; asked for b, it answers its count of a |
| HIGH | a, b and c | counts a and b |
| NEVER | b only | counts b; never saw `C a 40` |

**The fine-tunes.** Each trains a model to answer `C b 40` for 10K steps, read at 1K and 10K. A *spurious row* is a training row whose `count(b)` keeps its natural value and whose count of a in the window is set to `count(b) - 1` by swapping a's and c's, so on it the old count gives the right answer. The text holds only a, b and c, so the three window counts sum to 40 on every row.

| arm | base | learning rate | spurious rows | the alternative it tests |
|---|---|---|---|---|
| N1, N2, N3 | LOW | 1e-4 | none, half, 80% | learns b for real: the "new" side |
| M, M2, M3 | LOW | 1e-5 | none, half, 80% | slow: may pass through the old count first |
| W1, W2 | LOW | 1e-5, 1e-4 | all | the old count is right on every row: the "existing" side |
| R | HIGH | 1e-5 | none | already counts b: should run through that count |

N1 and W1 also ran at a second seed.

**The two checks**, on natural inputs where `count(b)` differs from `count(a) + 1`:

- **Behaviour**: for the count of a, the share of answers equal to `count(a) + 1`; for the count of b, the share equal to `count(b)`. 0 means never, 1 always.
- **Patch**: 1,000 pairs of inputs with the same other count and a different value of this one. We copy the donor's residual stream at `=` after block 5 into the other run. The patch is the share of pairs whose answer becomes the donor's. 0 means the swapped count is not used there; 1 means the answer always follows it.

Each check has a null and a positive reference, measured on the pretrained models before any fine-tune. The upper bar is halfway from null to positive; the lower bar sits just above the null. A state **goes through** a count when both checks clear the upper bar, **does not** when both sit under the lower bar, and reads **neither** otherwise.

**Possession** is read by a linear probe at `=` after each block. The model possesses a count when the probe beats its shuffled-label control by at least half of what LOW's count-a probe reaches. LOW's count-a margin also subtracts NEVER's count-a probe; the count-b margin has no such no-task control.


## What we expected, and why

- The pretrained models would behave as designed, and the patch would move answers on the models that use the swapped count and not on the nulls. The patch had never been tried on this grammar, so the lean was only medium.
- The fine-tunes would match the paper's count-b accuracies within 15 points in seven of nine cells. The two all-spurious cells could miss by construction: answering `count(a) + 1` scores about 18% on natural rows.
- The all-spurious fine-tunes would go through the count of a at both readings, N1 would not, and HIGH's fine-tune would go through its count of b. The paper says only what the all-spurious fine-tunes do not answer, so the lean was medium.
- LOW would possess a count of b at even odds: it almost never answers for b, but every task in the grammar counts tokens.


## What we observed

### Do the checks work on the pretrained models?

![The patch by block on the pretrained models](../experiments/pcfg_anchors/figures/controls.png)

**Reading it.** Yes. The figure plots, for each block whose output is swapped, the share of pairs whose answer becomes the donor's. On the two models asked for a count they use, the share rises to 0.97 and 1.00 at block 5. On the two nulls it is zero at every block. Block 5 is the earliest block near the best, so every fine-tune is read there. The caveat: block 5 is next to the output, so the patch there tests that the count reaches the answer late, not where it is first computed. The behaviour of the pretrained models also read as designed: LOW counts a at 99.98%, and asked for b it answers its count of a.


### Does the rebuilt grammar reproduce the paper's fine-tunes?

![Count-b accuracy at 10K against Table 6](../experiments/pcfg_anchors/figures/table6.png)

**Reading it.** Yes, in seven of nine cells, the most the filing allowed. Every fine-tune with any natural rows reaches at least 99.85% on count b, within six points of the paper. The two all-spurious fine-tunes miss. The slow one scores exactly what always answering `count(a) + 1` would, although by 10K it gives that answer on only 0.40 of the inputs where it is wrong: it earns its points on the rows where `count(b)` equals `count(a) + 1`. The fast one sits above that floor at 26%, where the paper prints 1.9%.

### Which fine-tunes run through the old count of a?

![Every fine-tuned state on the two checks](../experiments/pcfg_anchors/figures/labels.png)

**Reading it.** Fast fine-tunes never go through the count of a. Slow ones with many spurious rows go through it early, and by 10K neither check reads them as going through it. At 10K every fine-tune that learned to count b sits at zero on both checks and goes through the count of b.

The slow fine-tune with 80% spurious rows goes through the count of a at 1K: it gives `count(a) + 1` on 0.63 of inputs and follows a swapped count of a on 0.59 of pairs. By 10K it counts b and reads not through. The slow half-spurious fine-tune at 1K follows the swap on 0.52 of pairs, over the upper bar, but gives `count(a) + 1` on only 0.43, so it reads neither. The plan expected both to go through at 1K.

The all-spurious fine-tunes start where we expected: after 1K steps the slow one gives `count(a) + 1` on 0.88 of inputs and follows a swapped count of a on 0.72 of pairs. Its seed copy agrees. By 10K both copies sit between the bars and read neither. The fast all-spurious fine-tune reads neither at both readings, just under the upper behaviour bar at 1K. HIGH's fine-tune goes through its count of b.

The checks do not say what the slow all-spurious fine-tune computes at 10K. A swap at the last block shows that its answer depends on both counts, where at 1K it depended mostly on a. The text holds only a, b and c, so the three window counts sum to 40 on every row. That rules out an answer built from the count of c alone: any such answer that almost never equals `count(b)` gives `count(a) + 1` on at most 0.105 of inputs, and this state reads 0.40. One rule fits both of its numbers: the midpoint of `count(a) + 1` and `count(b)`, rounded towards `count(a) + 1`. It is right on every spurious row and reads the count of a on every input, through a function the behaviour check is blind to. It fits, but it is untested.

The two checks never pointed opposite ways on any state, so the filed null did not occur. Two readings were not filed. Every fine-tune from LOW that learned b also lost LOW's own count-a answer, as one counter re-bound to the new letter would. And of the sixteen states the instrument scoring would use, only two go through the count of a, where the plan expected six.

### Does LOW already hold a count of b?

![Probe margin by block](../experiments/pcfg_anchors/figures/probe.png)

**Reading it.** Yes by the filed rule, narrowly, and the reading behind it is weaker than the pass. LOW's count-b margin peaks at 0.277 in block 2, over the possession bar of 0.260 by about one and a half standard deviations, then falls towards the output. Its count-a margin rises to 0.52, and HIGH's count-b margin to 0.65.

The two margins subtract different controls. The count-a margin also subtracts NEVER's count-a probe, and NEVER, which never had a count-a task, still decodes count a at about half its rows, against a third for shuffled labels. The count-b margin subtracts shuffled labels only. Against a no-task control of that kind, LOW's count-b gap is under half the bar. The count of a alone also predicts the count of b on 0.40 of natural windows, so part of the margin can come from LOW's count of a. So the count of b decodes from LOW's residual above shuffled labels, by less than a model with no count-a task decodes the count of a. A probe also cannot say whether that count is used.

## What we learned

The reuse line has a ground truth that owes nothing to any decomposition, and its two checks never contradict each other here. On this grammar a fast fine-tune that learns a new count never runs its answer through the old count. Every slow fine-tune with 80% or more spurious rows runs through it early, and by the end neither check reads it as doing so. Whether the all-spurious ones still use the count of a at 10K, through some other function, the checks cannot say. So "existing" labels are scarce, two of eleven and both early, and any method scored against them, VPD included, faces a lopsided truth set: it must agree on ten of eleven states to pass.

## How compelling is the evidence

- The controls are clean: nulls at zero in every set and block, positives at 0.97 and 1.
- Each fine-tuned state is read once on 1,000 inputs and 1,000 pairs, so one reading has a sampling noise of about 0.016 near 0.5. Two cells sit within that of a bar; no count-a label depends on either.
- The behaviour check is blind to an answer that uses the count through another function, such as the midpoint rule. No positive control of the fine-tuned behaviour reading itself was run.
- The patch reads one block next to the output, at `=` only. The three counts sum to 40, so neither patch can tell a count of c from the count it swaps; the count-b labels carry that ambiguity too.
- The probe shows a readable count, not a used one; LOW's count of b clears its bar by little, against a weaker control than its count of a.
- One pretraining seed per model; second fine-tune seeds for N1 and W1 only, which agree on every count-a label; one task, one window.

## Surprises

- The all-spurious fine-tunes stop reading as going through the count of a between 1K and 10K, while still answering every spurious test row right. This is the filed miss; it is not a reversal.
- Every fine-tune from LOW that learned b also lost LOW's own count-a answer.
- Before any fine-tune, LOW's count-b accuracy equals exactly the score of answering its count of a.

## What next

1. **Decide whether instrument scoring goes ahead on eleven labelled states, two of them "existing".** The plan's decision tree does not cover a miss without reversal. If the truth set is judged too lopsided, more "existing" anchors come first.
2. **Read the per-row answers of the all-spurious states at 10K against the counts of a and b**, free on the saved states; E55 Amendment 1 is being written for this. If they follow the midpoint rule, those states still use the count of a and the truth set may gain "existing" anchors; if not, the "neither" labels stand.
3. **Read the all-spurious arms between 1K and 10K**, to find when they stop going through the count of a and save more "existing" states.

## Scorecard

| id | claim in words | rule | read | verdict |
|---|---|---|---|---|
| PA1 | pretrained models and controls read as tabled | every behaviour, probe, patch and null row | all rows clear; patch 0.966 and 1.000, nulls 0 | pass |
| PA2 | count-b accuracy at 10K within 15 points of Table 6 | at least 7 of 9 cells | 7 of 9; all-spurious cells off by 17.7 and 24.05 | pass |
| PA3 | all-spurious arms through count a at both readings; N1 not; R through count b | all 7 readings | 4 of 7: slow all-spurious 10K and fast all-spurious both read neither | **miss**, not reversed |
| PA4 | LOW possesses the count of b | margin at least 3 sd and half of LOW's count-a margin | 0.277 against 0.260 and 0.032, block 2 | pass by its rule, narrowly; uneven controls |

## Not run

- Stage B on the 67M, the free instrument stage, and the gated VPD decomposition of LOW.
- Per-row answers of the all-spurious states at 10K, which would test the midpoint rule; states between 1K and 10K; probes on fine-tuned states.
- A mechanism-level test separating the count of c from the counts of a and b; a patch at any position other than `=`.
- A positive control of the fine-tuned behaviour reading; a count-b patch null on `C b 40`; resampling of each state's readings; a no-task control for the count-b probe.
- A second pretraining seed; more fine-tune seeds; the paper's 200K pretraining.

Provenance: `experiments/pcfg_anchors/results/` (`a1_*.json`, `a1c_patch.json`, `a1c_probe.json`, `a2_*.json`), code `experiments/pcfg_anchors/run.py` and `src/commute/interchange.py`. MATS cluster debug QoS: pretrainings 44268-44270, controls 44306 and 44307, fine-tunes 44328-44330; $0.